# Initial Setup and Data Loading

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load Raw ANNOVAR Data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/AI_WES_AD_Project/ANNOVAR result/AD_Cohort_111_2.hg38_multianno.csv")

print("Original shape:", df.shape)

Original shape: (331161, 139)


## 3. Initial Data Exploration

In [ ]:
print("Total variants:", df.shape)

print("\nFunctional class distribution:")
print(df["Func.refGene"].value_counts())

print("\nExonic function distribution:")
print(df["ExonicFunc.refGene"].value_counts())

print("\ngnomAD distribution:")
print(df["gnomAD_exome_ALL"].describe())


Total variants: (331161, 139)

Functional class distribution:
Func.refGene
exonic                   237051
intronic                  49337
UTR5                      11323
intergenic                 8781
splicing                   5964
ncRNA_intronic             5678
UTR3                       5444
ncRNA_exonic               3363
upstream                   2769
downstream                  656
ncRNA_splicing              394
upstream;downstream         173
exonic;splicing             156
ncRNA_exonic;splicing        44
UTR5;UTR3                    28
Name: count, dtype: int64

Exonic function distribution:
ExonicFunc.refGene
nonsynonymous SNV             137539
.                              93954
synonymous SNV                 78416
stopgain                        6379
frameshift deletion             5341
nonframeshift substitution      4707
nonframeshift deletion          1270
frameshift insertion             953
unknown                          768
nonframeshift insertion          631

## 4. Convert Frequency Columns to Numeric

In [ ]:
# Convert frequency columns to numeric
df["gnomAD_exome_ALL"] = pd.to_numeric(df["gnomAD_exome_ALL"], errors="coerce")
df["gnomAD_genome_ALL"] = pd.to_numeric(df["gnomAD_genome_ALL"], errors="coerce")

print("\nMAF column summary:")
print(df[["gnomAD_exome_ALL","gnomAD_genome_ALL"]].describe())


MAF column summary:
       gnomAD_exome_ALL  gnomAD_genome_ALL
count     135645.000000      127284.000000
mean           0.123350           0.151507
std            0.225522           0.240052
min            0.000000           0.000000
25%            0.000200           0.001800
50%            0.009500           0.025000
75%            0.129800           0.199700
max            1.000000           1.000000


## 5. Filtering for Rare Variants

#🧬 Biological reasoning: exome vs genome filtering
🧠gnomAD_exome:
*   Based on exome sequencing
*   More coverage in coding regions
*   Larger effective sample size for coding variants
*   More reliable AF for exonic variants

🧠gnomAD_genome:
*   Based on whole genome sequencing
*   Includes intronic/intergenic
*   Slightly different population composition
*   Sometimes smaller coverage in coding compared to exome dataset

##Apply MAF < 0.01 filter


In [ ]:
maf_filtered = df[
    ((df["gnomAD_exome_ALL"].isna()) | (df["gnomAD_exome_ALL"] < 0.01)) &
    ((df["gnomAD_genome_ALL"].isna()) | (df["gnomAD_genome_ALL"] < 0.01))
]

print("\nAfter MAF filtering:")
print("Shape:", maf_filtered.shape)




After MAF filtering:
Shape: (251551, 139)


## 6. Save MAF Filtered Data

###Check Reduction Percentage

In [ ]:
reduction = 100 * (1 - maf_filtered.shape[0] / df.shape[0])
print(f"Dataset reduced by {reduction:.2f}%")


Dataset reduced by 24.04%


In [ ]:
#Save Filtered File
maf_filtered.to_csv("AD_MAF_filtered.csv", index=False)


### 6.1. Copy to Google Drive

In [ ]:
import shutil
import os

source_file = "AD_MAF_filtered.csv"
destination_path = "/content/drive/MyDrive/AI_WES_AD_Project/ANNOVAR result"

# Create the destination directory if it doesn't exist
os.makedirs(destination_path, exist_ok=True)

# Copy the file
shutil.copy(source_file, destination_path)
print(f"File '{source_file}' copied to '{destination_path}'")

File 'AD_MAF_filtered.csv' copied to '/content/drive/MyDrive/AI_WES_AD_Project/ANNOVAR result'


## 7. Functional Region Filtering

##Functional Region Filter
*   Keep only coding relevant regions.

This removes intronic, intergenic, upstream/downstream, ncRNA variants

In [ ]:
func_filtered = maf_filtered[
    maf_filtered["Func.refGene"].isin(["exonic", "splicing"])
]

print("\nAfter functional region filtering:")
print("Shape:", func_filtered.shape)

print("\nFunctional distribution:")
print(func_filtered["Func.refGene"].value_counts())

print("\nExonic function distribution:")
print(df["ExonicFunc.refGene"].value_counts())


After functional region filtering:
Shape: (184690, 139)

Functional distribution:
Func.refGene
exonic      179011
splicing      5679
Name: count, dtype: int64

Exonic function distribution:
ExonicFunc.refGene
nonsynonymous SNV             137539
.                              93954
synonymous SNV                 78416
stopgain                        6379
frameshift deletion             5341
nonframeshift substitution      4707
nonframeshift deletion          1270
frameshift insertion             953
unknown                          768
nonframeshift insertion          631
startloss                        606
frameshift substitution          319
stoploss                         278
Name: count, dtype: int64


## 8. Exonic Impact Filtering

##Exonic Impact Filter
*   Keep protein-altering variants.

In [ ]:
impact_variants = [
    "nonsynonymous SNV",
    "stopgain",
    "stoploss",
    "frameshift insertion",
    "frameshift deletion",
    "frameshift substitution",
    "startloss"
]

impact_filtered = func_filtered[
    (func_filtered["Func.refGene"] == "splicing") |
    (func_filtered["ExonicFunc.refGene"].isin(impact_variants))
]

print("\nAfter impact filtering:")
print("Shape:", impact_filtered.shape)

print("\nImpact distribution:")
print(impact_filtered["ExonicFunc.refGene"].value_counts())


After impact filtering:
Shape: (128843, 139)

Impact distribution:
ExonicFunc.refGene
nonsynonymous SNV          109962
stopgain                     6135
.                            5679
frameshift deletion          5146
frameshift insertion          838
startloss                     528
frameshift substitution       319
stoploss                      236
Name: count, dtype: int64


## 9. Save Impact Filtered Data

In [ ]:
impact_filtered.to_csv("AD_MAF_impact_filtered.csv", index=False)

### 9.1. Copy to Google Drive

In [ ]:
import shutil
import os

source_file = "AD_MAF_impact_filtered.csv"
destination_path = "/content/drive/MyDrive/AI_WES_AD_Project/ANNOVAR result"

# Create the destination directory if it doesn't exist
os.makedirs(destination_path, exist_ok=True)

# Copy the file
shutil.copy(source_file, destination_path)
print(f"File '{source_file}' copied to '{destination_path}'")

File 'AD_MAF_impact_filtered.csv' copied to '/content/drive/MyDrive/AI_WES_AD_Project/ANNOVAR result'


## 10. Prepare Data for Feature Engineering

In [ ]:
print("Original shape:", impact_filtered.shape)

Original shape: (128843, 139)


## 11. Feature Selection

In [ ]:
selected_columns = [
    "Chr", "Start", "End", "Ref", "Alt",
    "Gene.refGene","Func.refGene","ExonicFunc.refGene",
    "gnomAD_exome_ALL", "gnomAD_genome_ALL",
    "SIFT_score",
    "Polyphen2_HDIV_score",
    "Polyphen2_HVAR_score",
    "LRT_score",
    "MutationTaster_score",
    "DANN_score",
    "CADD_phred",
    "FATHMM_score",
    "PROVEAN_score",
    "fathmm-MKL_coding_score",
    "fathmm-XF_coding_score",
    "MetaSVM_score",
    "MetaLR_score",
    "REVEL_score",
    "M-CAP_score",
    "VEST4_score",
    "Eigen-raw_coding",
    "ClinPred_score",
    "GERP++_RS",
    "phyloP100way_vertebrate",
    "phastCons100way_vertebrate",
    "LINSIGHT",
    "SiPhy_29way_logOdds",
    "CLNSIG"
]

structured_df = impact_filtered[selected_columns]

print("Shape of structured_df:", structured_df.shape)
print("\nFirst 5 rows of structured_df:")
display(structured_df.head())

Shape of structured_df: (128843, 34)

First 5 rows of structured_df:


,Chr,Start,End,Ref,Alt,Gene.refGene,Func.refGene,ExonicFunc.refGene,gnomAD_exome_ALL,gnomAD_genome_ALL,...,M-CAP_score,VEST4_score,Eigen-raw_coding,ClinPred_score,GERP++_RS,phyloP100way_vertebrate,phastCons100way_vertebrate,LINSIGHT,SiPhy_29way_logOdds,CLNSIG
5,chr1,69503,69503,T,A,OR4F5,exonic,nonsynonymous SNV,NaN,NaN,...,0.007,0.442,-0.707,0.469,1.17,0.804,0.002,.,4.977,.
80,chr1,924514,924514,T,C,SAMD11,exonic,nonsynonymous SNV,NaN,NaN,...,.,.,.,.,2.29,1.270,0.065,0.168,6.532,.
81,chr1,924517,924517,C,T,SAMD11,exonic,nonsynonymous SNV,NaN,NaN,...,.,.,.,.,2.29,1.330,0.199,0.168,12.183,.
83,chr1,924538,924538,A,C,SAMD11,exonic,nonsynonymous SNV,NaN,NaN,...,.,.,.,.,2.29,3.675,1.000,0.730,9.663,.
84,chr1,924564,924564,G,T,SAMD11,exonic,nonsynonymous SNV,NaN,NaN,...,.,.,.,.,2.37,4.499,1.000,0.145,4.218,.


## 12. Save Structured Data

In [ ]:
structured_df.to_csv("/content/drive/MyDrive/AI_WES_AD_Project/Feature Eng/AD_Cohort_Extract_feature.csv", index=False)

print("Structured dataset saved successfully to Google Drive.")

Structured dataset saved successfully to Google Drive.


## 13. Audit Removed Columns

In [ ]:
removed_columns = list(set(df.columns) - set(selected_columns))

print("Total removed columns:", len(removed_columns))
for col in sorted(removed_columns):
    print(col)

Total removed columns: 105
AAChange.refGene
Aloft_Confidence
Aloft_pred
BayesDel_addAF_pred
BayesDel_addAF_rankscore
BayesDel_addAF_score
BayesDel_noAF_pred
BayesDel_noAF_rankscore
BayesDel_noAF_score
CADD_raw
CADD_raw_rankscore
CLNALLELEID
CLNDISDB
CLNDN
CLNREVSTAT
ClinPred_pred
ClinPred_rankscore
DANN_rankscore
DEOGEN2_pred
DEOGEN2_rankscore
DEOGEN2_score
Eigen-PC-raw_coding
Eigen-PC-raw_coding_rankscore
Eigen-raw_coding_rankscore
FATHMM_converted_rankscore
FATHMM_pred
GERP++_NR
GERP++_RS_rankscore
GTEx_V8_gene
GTEx_V8_tissue
GeneDetail.refGene
GenoCanyon_rankscore
GenoCanyon_score
Interpro_domain
LINSIGHT_rankscore
LIST-S2_pred
LIST-S2_rankscore
LIST-S2_score
LRT_converted_rankscore
LRT_pred
M-CAP_pred
M-CAP_rankscore
MPC_rankscore
MPC_score
MVP_rankscore
MVP_score
MetaLR_pred
MetaLR_rankscore
MetaRNN_pred
MetaRNN_rankscore
MetaRNN_score
MetaSVM_pred
MetaSVM_rankscore
MutPred_rankscore
MutPred_score
MutationAssessor_pred
MutationAssessor_rankscore
MutationAssessor_score
MutationTast

## 14. Missing Value Analysis

In [ ]:
df = structured_df
print("Original shape:", df.shape)

Original shape: (128843, 34)


### 14.1. Check NaN Count in Population Frequency Columns

In [ ]:
print("NaN count in population frequency columns:\n")

maf_columns = ["gnomAD_exome_ALL", "gnomAD_genome_ALL"]

for col in maf_columns:
    print(f"{col}: {df[col].isna().sum()} NaN values")

NaN count in population frequency columns:

gnomAD_exome_ALL: 93811 NaN values
gnomAD_genome_ALL: 106417 NaN values


### 14.2. Define Score Columns for Analysis

In [ ]:
score_columns = [
"SIFT_score",
    "Polyphen2_HDIV_score",
    "Polyphen2_HVAR_score",
    "LRT_score",
    "MutationTaster_score",
    "DANN_score",
    "CADD_phred",
    "FATHMM_score",
    "PROVEAN_score",
    "fathmm-MKL_coding_score",
    "fathmm-XF_coding_score",
    "MetaSVM_score",
    "MetaLR_score",
    "REVEL_score",
    "M-CAP_score",
    "VEST4_score",
    "Eigen-raw_coding",
    "ClinPred_score",
    "GERP++_RS",
    "phyloP100way_vertebrate",
    "phastCons100way_vertebrate",
    "LINSIGHT",
    "SiPhy_29way_logOdds",
    "CLNSIG"
]

### 14.3. Check Dot ('.') Count in Prediction Score Columns

In [ ]:
print("\nDot ('.') count in prediction score columns:\n")

for col in score_columns:
    dot_count = (df[col] == ".").sum()
    print(f"{col}: {dot_count} '.' values")


Dot ('.') count in prediction score columns:

SIFT_score: 27713 '.' values
Polyphen2_HDIV_score: 33729 '.' values
Polyphen2_HVAR_score: 33729 '.' values
LRT_score: 37711 '.' values
MutationTaster_score: 13390 '.' values
DANN_score: 10035 '.' values
CADD_phred: 8029 '.' values
FATHMM_score: 32300 '.' values
PROVEAN_score: 27180 '.' values
fathmm-MKL_coding_score: 10036 '.' values
fathmm-XF_coding_score: 22759 '.' values
MetaSVM_score: 24205 '.' values
MetaLR_score: 24205 '.' values
REVEL_score: 24205 '.' values
M-CAP_score: 29197 '.' values
VEST4_score: 14441 '.' values
Eigen-raw_coding: 18462 '.' values
ClinPred_score: 24011 '.' values
GERP++_RS: 11053 '.' values
phyloP100way_vertebrate: 8030 '.' values
phastCons100way_vertebrate: 8030 '.' values
LINSIGHT: 121558 '.' values
SiPhy_29way_logOdds: 12641 '.' values
CLNSIG: 116883 '.' values


### 14.4. Calculate Missing Percentages

## Calculate Missing Percentages
Calculate the missing percentages (including both NaN and '.' values) for each column in the `score_columns` list and store them for visualization.


In [ ]:
missing_percentages = {}
total_rows = len(df)

for col in score_columns:
    nan_count = df[col].isna().sum()
    dot_count = (df[col] == ".").sum()

    total_missing_count = nan_count + dot_count
    missing_percent = (total_missing_count / total_rows) * 100

    missing_percentages[col] = missing_percent

print("Missing percentages stored in 'missing_percentages' dictionary:")
for col, percent in missing_percentages.items():
    print(f"{col}: {percent:.2f}%")

Missing percentages stored in 'missing_percentages' dictionary:
SIFT_score: 21.51%
Polyphen2_HDIV_score: 26.18%
Polyphen2_HVAR_score: 26.18%
LRT_score: 29.27%
MutationTaster_score: 10.39%
DANN_score: 7.79%
CADD_phred: 6.23%
FATHMM_score: 25.07%
PROVEAN_score: 21.10%
fathmm-MKL_coding_score: 7.79%
fathmm-XF_coding_score: 17.66%
MetaSVM_score: 18.79%
MetaLR_score: 18.79%
REVEL_score: 18.79%
M-CAP_score: 22.66%
VEST4_score: 11.21%
Eigen-raw_coding: 14.33%
ClinPred_score: 18.64%
GERP++_RS: 8.58%
phyloP100way_vertebrate: 6.23%
phastCons100way_vertebrate: 6.23%
LINSIGHT: 94.35%
SiPhy_29way_logOdds: 9.81%
CLNSIG: 90.72%


## 15. Dataset Refinement and Cleaning

In [ ]:
df = structured_df
print("Original shape:", df.shape)

Original shape: (128843, 34)


### 15.1. Drop Columns with High Missingness

In [ ]:
df = df.drop(columns=[
"gnomAD_genome_ALL",
"LINSIGHT"
], errors='ignore')

### 15.2. Check Shape After Dropping Columns

In [ ]:
df.shape

(128843, 32)

### 15.3. Remove Rows with Any Missing Values

In [ ]:
print("Shape of df before dropping NaNs:", df.shape)

# Remove rows with any NaN values
df_cleaned = df.dropna()

print("Shape of df_cleaned after dropping NaNs:", df_cleaned.shape)

Shape of df before dropping NaNs: (128843, 32)
Shape of df_cleaned after dropping NaNs: (35032, 32)


## 16. Final Export of Cleaned Feature Set

In [ ]:
df_cleaned.to_csv("/content/drive/MyDrive/AI_WES_AD_Project/Feature Eng/AD_df_cleaned_feature.csv", index=False)

print("Structured dataset saved successfully to Google Drive.")

Structured dataset saved successfully to Google Drive.
